In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    (candidate for candidate in (Path.cwd(), *Path.cwd().parents) if (candidate / "notebook_setup.py").exists()),
    Path.cwd(),
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from notebook_setup import get_dataset_path

import numpy as np
import scipy.io as sio
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize, LinearSegmentedColormap
from matplotlib.cm import ScalarMappable

# =============================================================================
# =============================================================================
plt.rcParams["font.family"] = "Arial"
plt.rcParams["font.size"] = 20

mat_path = get_dataset_path("CETL_data_processed.mat")
data = sio.loadmat(mat_path)

processed_Qd_mat  = data["processed_Qd_mat"].astype(float)
processed_Vd_mat  = data["processed_Vd_mat"].astype(float)

n_rows, n_cells = processed_Qd_mat.shape
Qmax_all = processed_Qd_mat[-1, :].ravel()
n_samples = processed_Qd_mat.shape[1]

# -----------------------------------------------------------------------------
# -----------------------------------------------------------------------------
nature_colors = [
    "#313695",
    "#74add1",
    "#e0f3f8",
    "#fee090",
    "#f46d43",
    "#a50026",
]
cmap_nature = LinearSegmentedColormap.from_list("Nature_Style", nature_colors, N=256)
norm_nature = Normalize(vmin=np.nanmin(Qmax_all), vmax=np.nanmax(Qmax_all))

# =============================================================================
# =============================================================================
Vtail_upp_list = [2.73, 2.75, 2.77]

for V_tail_upp in Vtail_upp_list:
    print(f"Generating plots for V_tail_upp = {V_tail_upp} V...")
    
    V_low, V_high = 2.7, 3.0
    mu, sigma = (V_low + V_high) / 2.0, (V_high - V_low) / 3.0
    rng = np.random.default_rng(999)
    V_init = np.empty(n_cells, dtype=float)
    
    for i in range(n_cells):
        while True:
            v = rng.normal(mu, sigma)
            if v > V_tail_upp:
                V_init[i] = v
                break

    # =========================================================================
    # =========================================================================
    fig1 = plt.figure(figsize=(15, 5), constrained_layout=True)
    subfigs1 = fig1.subfigures(1, 2, width_ratios=[1, 1], wspace=0.1)

    # -------------------------------------------------------------------------
    # -------------------------------------------------------------------------
    ax_tail = subfigs1[0].subplots(1, 1)
    Vtail_low = 2.50

    for i in range(n_samples):
        Vd = processed_Vd_mat[:, i]
        Qd = processed_Qd_mat[:, i]
        mask = (Vd >= Vtail_low) & (Vd <= V_tail_upp)
        if not np.any(mask): continue
        ax_tail.plot(Qd[mask], Vd[mask], color=cmap_nature(norm_nature(Qmax_all[i])), linewidth=1.3, alpha=0.9)

    ax_tail.set_xlabel("Remaining discharge capacity (Ah)", labelpad=10)
    ax_tail.set_ylabel("Voltage (V)", labelpad=10)
    ax_tail.grid(False)

    sm = ScalarMappable(norm=norm_nature, cmap=cmap_nature)
    sm.set_array([])
    cbar_tail = subfigs1[0].colorbar(sm, ax=ax_tail, pad=0.06)
    cbar_tail.set_label("Maximum capacity (Ah)", labelpad=10)
    cbar_tail.outline.set_visible(False)

    # -------------------------------------------------------------------------
    # -------------------------------------------------------------------------
    SOC_init = np.full(n_cells, np.nan, dtype=float)
    SOC_ref = np.full(n_cells, np.nan, dtype=float)
    V_ref = V_tail_upp

    for i in range(n_cells):
        Vd_i = processed_Vd_mat[:, i]
        Qd_i = processed_Qd_mat[:, i]
        v0   = V_init[i]
        
        valid_mask = np.isfinite(Vd_i) & np.isfinite(Qd_i)
        if not np.any(valid_mask): continue
        
        Vd_valid, Qd_valid = Vd_i[valid_mask], Qd_i[valid_mask]
        v_min, v_max = Vd_valid.min(), Vd_valid.max()
        
        sort_idx = np.argsort(Vd_valid)
        Vd_sorted, Qd_sorted = Vd_valid[sort_idx], Qd_valid[sort_idx]
        
        if min(v_min, v_max) <= v0 <= max(v_min, v_max) and np.isfinite(Qmax_all[i]) and Qmax_all[i] > 0:
            Q_init = np.interp(v0, Vd_sorted, Qd_sorted)
            SOC_init[i] = Q_init / Qmax_all[i]

        if min(v_min, v_max) <= V_ref <= max(v_min, v_max) and np.isfinite(Qmax_all[i]) and Qmax_all[i] > 0:
            Q_ref = np.interp(V_ref, Vd_sorted, Qd_sorted)
            SOC_ref[i] = Q_ref / Qmax_all[i]

    valid_init, valid_ref = np.isfinite(SOC_init), np.isfinite(SOC_ref)
    soc_init_percent, soc_ref_percent = SOC_init[valid_init] * 100.0, SOC_ref[valid_ref] * 100.0

    if soc_init_percent.size > 0 and soc_ref_percent.size > 0:
        all_soc_percent = np.concatenate([soc_init_percent, soc_ref_percent])
        bins = np.histogram_bin_edges(all_soc_percent, bins=30)
        
        counts_init, bin_edges = np.histogram(soc_init_percent, bins=bins)
        counts_ref,  _         = np.histogram(soc_ref_percent,  bins=bins)
        
        bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
        bin_width   = (bin_edges[1] - bin_edges[0]) * 0.8 

        ax_soc_top, ax_soc_bot = subfigs1[1].subplots(2, 1, sharex=True, gridspec_kw={"hspace": 0})
        
        color_init, color_ref = "#4575b4", "#d73027" 
        
        ax_soc_top.bar(bin_centers, counts_init, width=bin_width, color=color_init, alpha=0.85, edgecolor="black", linewidth=0.6, label="Initial SOC")
        ax_soc_top.set_ylabel("Cell number", labelpad=10)
        ax_soc_top.grid(False)
        ax_soc_top.tick_params(labelbottom=False)
        ax_soc_top.legend(frameon=False, loc="upper left", fontsize=16)
        
        ax_soc_bot.bar(bin_centers, counts_ref, width=bin_width, color=color_ref, alpha=0.85, edgecolor="black", linewidth=0.6, label=f"SOC at {V_tail_upp} V")
        ax_soc_bot.set_xlabel("SOC (%)", labelpad=10)
        ax_soc_bot.set_ylabel("Cell number", labelpad=10)
        ax_soc_bot.grid(False)
        ax_soc_bot.legend(frameon=False, loc="upper right", fontsize=16)

        for ax in [ax_soc_top, ax_soc_bot]:
            for spine in ["top", "right", "left", "bottom"]:
                ax.spines[spine].set_visible(True)

        max_count = max(counts_init.max(), counts_ref.max())
        ax_soc_top.set_yticks(np.arange(0, max_count + 300, 200))
        ax_soc_bot.set_yticks(np.arange(0, max_count + 300, 1200))
        
        ax_soc_top.set_ylim(0, max(600, counts_init.max() * 1.15))
        ax_soc_bot.set_ylim(0, max(4100, counts_ref.max() * 1.15))

    fig1.savefig(f"fig1_tail_curve_and_SOC_Vtail_{V_tail_upp}.png", dpi=300, bbox_inches="tight")
    plt.close(fig1)


    # =========================================================================
    # =========================================================================
    fig2, axes2 = plt.subplots(2, 3, figsize=(15, 5), sharex=False, sharey=False, constrained_layout=True)
    axes2_flat = axes2.ravel()

    Vseg_low = V_tail_upp
    Vseg_upp = V_tail_upp + 0.30
    nun_seg = 6
    V_edges = np.linspace(Vseg_low, Vseg_upp, nun_seg + 1)

    for seg_idx in range(nun_seg):
        ax = axes2_flat[seg_idx]
        V_low_k = V_edges[seg_idx]
        V_upp_k = V_edges[seg_idx + 1]

        for i in range(n_samples):
            Vd = processed_Vd_mat[:, i]
            Qd = processed_Qd_mat[:, i]
            mask = (Vd >= V_low_k) & (Vd <= V_upp_k)
            if not np.any(mask): continue
            
            ax.plot(Qd[mask], Vd[mask], color=cmap_nature(norm_nature(Qmax_all[i])), linewidth=1.0, alpha=0.9)

        ax.grid(False)
        ax.autoscale(enable=True, axis="both", tight=False)

    fig2.supxlabel("Remaining discharge capacity (Ah)", fontsize=20, y=-0.06)
    fig2.supylabel("Voltage (V)", fontsize=20, x=-0.03)

    cbar_seg = fig2.colorbar(sm, ax=axes2, pad=0.03, fraction=0.06)
    cbar_seg.set_label("Maximum capacity (Ah)", labelpad=15)
    cbar_seg.outline.set_visible(False)

    fig2.savefig(f"fig2_upstream_segment_curves_2x3_Vtail_{V_tail_upp}.png", dpi=300, bbox_inches="tight")
    plt.close(fig2)

print("\nAll figures have been generated and saved.")
